# Day 8 — LangGraph Foundations

In this module, we transition from sequential loops to structured state machines. We use LangGraph to model applications as graphs composed of nodes (Python functions) and edges (transitions), enabling precise routing and state control.

## Objectives
- Build a compiled graph structure using StateGraph.
- Configure sequential flow paths and conditional edge routing.
- **Extended Implementation:** Build a Support Ticket Triage Router graph, implement a state transition auditor, and configure memory checkpointer state persistence.

In [ ]:
# Install dependencies
!pip install langgraph langchain-groq python-dotenv -q
print("✅ Libraries installed.")

## 1. The LangGraph Concept
A graph structure contains:
- **State:** The shared memory dictionary passed through nodes.
- **Nodes:** Workstation functions that process and return state updates.
- **Edges:** Directions detailing which node executes next.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END

# Define the State
class SimpleState(TypedDict):
    message: str
    history: list

# Define Nodes
def node_greet(state: SimpleState):
    print("Entering: Greet Node")
    return {"message": f"Hello! {state['message']}", "history": ["greet"]}

def node_signoff(state: SimpleState):
    print("Entering: Signoff Node")
    return {"message": f"{state['message']} Have a great day!", "history": state['history'] + ["signoff"]}

# Assemble Graph
builder = StateGraph(SimpleState)
builder.add_node("greet", node_greet)
builder.add_node("signoff", node_signoff)

builder.set_entry_point("greet")
builder.add_edge("greet", "signoff")
builder.add_edge("signoff", END)

graph = builder.compile()
print("✅ Graph compiled successfully.")

In [ ]:
# Run Graph
output = graph.invoke({"message": "Welcome to CareerForge learning module.", "history": []})
print("\nFinal Output:", output)

---
# Extended Implementation
Below are implementation techniques covering ticket routing graphs, history auditing, and checkpointers.

### 1. Support Ticket Triage Router Graph
We construct a multi-node ticket routing graph that classifies client queries (Billing, Technical, General), routes requests to specialized workers, and compiles a final support message.

In [ ]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=os.getenv("GROQ_API_KEY"), temperature=0)

# Graph State definition
class TicketState(TypedDict):
    query: str
    category: str
    draft: str
    final_response: str
    audit_log: list

# Node 1: Classifier
def classify_ticket(state: TicketState):
    q = state['query']
    prompt = f"Classify this support query as 'billing', 'technical', or 'general'. Respond in exactly one word. Query: {q}"
    category = llm.invoke(prompt).content.strip().lower()
    return {
        "category": category,
        "audit_log": state.get('audit_log', []) + [f"Classified as {category}"]
    }

# Node 2: Billing specialist
def handle_billing(state: TicketState):
    return {
        "draft": "Billing inquiry routed. Please note refunds take 3-5 business days to process.",
        "audit_log": state['audit_log'] + ["Handled by Billing Dept"]
    }

# Node 3: Tech support specialist
def handle_tech(state: TicketState):
    return {
        "draft": "Technical issue logged. Please try restarting your device or checking your internet connection.",
        "audit_log": state['audit_log'] + ["Handled by Tech Support Dept"]
    }

# Node 4: General support
def handle_general(state: TicketState):
    return {
        "draft": "General query received. A customer representative will reply to you within 24 hours.",
        "audit_log": state['audit_log'] + ["Handled by General Dept"]
    }

# Node 5: Summarizer
def finalize_response(state: TicketState):
    final_str = f"[Response Draft]\n{state['draft']}\n\nThank you for contacting support!"
    return {
        "final_response": final_str,
        "audit_log": state['audit_log'] + ["Response finalized"]
    }

# Conditional Router
def route_ticket(state: TicketState):
    cat = state['category']
    if "billing" in cat:
        return "billing_node"
    elif "technical" in cat:
        return "tech_node"
    else:
        return "general_node"

# Build StateGraph
ticket_graph = StateGraph(TicketState)
ticket_graph.add_node("classifier", classify_ticket)
ticket_graph.add_node("billing_node", handle_billing)
ticket_graph.add_node("tech_node", handle_tech)
ticket_graph.add_node("general_node", handle_general)
ticket_graph.add_node("finalizer", finalize_response)

ticket_graph.set_entry_point("classifier")
ticket_graph.add_conditional_edges(
    "classifier",
    route_ticket,
    {
        "billing_node": "billing_node",
        "tech_node": "tech_node",
        "general_node": "general_node"
    }
)
ticket_graph.add_edge("billing_node", "finalizer")
ticket_graph.add_edge("tech_node", "finalizer")
ticket_graph.add_edge("general_node", "finalizer")
ticket_graph.add_edge("finalizer", END)

compiled_ticket_graph = ticket_graph.compile()
print("✅ Router graph compiled successfully.")

### 2. Testing Router Graph & Auditing State History

In [ ]:
def run_ticket_test(query_str):
    initial_state = {
        "query": query_str,
        "category": "",
        "draft": "",
        "final_response": "",
        "audit_log": []
    }
    
    res = compiled_ticket_graph.invoke(initial_state)
    print(f"\nQUERY: '{query_str}'")
    print(f"RESPONSE:\n{res['final_response']}")
    print(f"AUDIT LOG: {res['audit_log']}")

run_ticket_test("I was charged twice on my credit card this month!")
run_ticket_test("My connection is extremely slow, and I can't load the login page.")

### 3. Pausable Checkpointer State Persistence Demo
We compile our graph alongside checkpointer memory to show how state changes are persisted and can be queried or resumed.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Create checkpoint database in memory
memory = MemorySaver()

# Compile graph with checkpointer
persisted_graph = ticket_graph.compile(checkpointer=memory)
config = {"configurable": {"thread_id": "test_thread_1"}}

# Invoke graph on thread
state_res = persisted_graph.invoke({"query": "General inquiry check", "audit_log": []}, config)
print("State Checkpointed:", state_res['audit_log'])

# Retrieve checkpointed state
snapshot = persisted_graph.get_state(config)
print("Retrieved Snapshot Next Node:", snapshot.next)
print("Retrieved Snapshot Values:", snapshot.values)